# 04 — Fixed depth pruning (HOOK C)

Skip **N decoder layers** of the LLM entirely: no computation, hidden
states pass straight through to the next layer. No retraining, no
fine-tuning, no change to the weights that remain.

**This is the only one of the three that makes a model call cheaper.**
Profiling a UniVLA control step gives 6% VQ encoding / 13% prefill / **70%
autoregressive decode**, and decode pays for every layer on every generated
token. Anything on the visual path is capped at ~19% no matter how
aggressive it is; removing layers is what actually moves wall-clock.

That 70% is a UniVLA measurement, not a constant. **Profile the target
backbone before assuming the same payoff** — the share of time spent in the
decoder stack is what sets the ceiling here, and it differs a lot by
architecture (see the assumptions section below).

**"Fixed"** means the same N layers stay bypassed for the whole episode.
(A phase-adaptive variant that changes N mid-episode exists and is
deliberately not in this notebook — it is a separate condition.)

## The method in three steps

1. **Measure**, once, how much each layer changes the representation:
   `1 − cos(layer_input, layer_output)`. Small means the layer barely moves
   the hidden state — it is idle.
2. **Rank** and pick the N most idle, subject to two safeguards (below).
3. **Replace** those modules with a pass-through.

Step 1 rides on the first inference of the episode, which has to run
anyway, so calibration costs **no extra forward pass**.

### The two safeguards, and why each exists

| safeguard | rule | why |
|---|---|---|
| **protect the early stack** | only the back half is eligible | Early layers perform the foundational transforms everything downstream depends on. Measured on Gemma2: bypassing layers 2 and 4 made generation never terminate. |
| **enforce a gap** | no two bypassed layers adjacent | Consecutive removals compound — the second layer's input is already wrong — so a gap-respecting greedy pass runs first, then the remainder fills in. |

Both thresholds (`min_layer=0.5`, `min_gap=1`) are **heuristics tuned on the
backbones we ran**, not derived quantities. They are exposed as arguments
rather than hard-coded so a new backbone can be swept. What must not change
between backbones is the *rule*: the claim "backbone A tolerates more depth
removal than backbone B" only means something if both were ranked and cut
identically.

## The pass-through layer

The subtle part is not skipping the computation; it is the **KV cache**.

`transformers`' `DynamicCache` indexes by `layer_idx`. A layer that never
calls `cache.update()` leaves a gap in that list, and a *later* layer's
update then raises `IndexError: list index out of range`. So the bypass
still writes a correctly-shaped zero placeholder. It is never read — this
layer has no attention — so zeros are safe.

Getting this wrong does not always crash. It can instead silently corrupt
the cache, which shows up only as a lower success rate — indistinguishable
from "the method does not work". The check at the end of this notebook
exists for that reason.

In [ ]:
from typing import Any, Optional

import torch


class BypassDecoderLayer(torch.nn.Module):
    """Skips the layer's compute, writes a placeholder KV to keep the cache
    contiguous."""

    def __init__(self, layer_idx: int, num_kv_heads: Optional[int] = None,
                 head_dim: Optional[int] = None):
        super().__init__()
        self.layer_idx = int(layer_idx)
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim

    def forward(self, hidden_states, attention_mask=None, position_ids=None,
                past_key_value=None, output_attentions=False, use_cache=False,
                **kwargs):
        if (use_cache and past_key_value is not None
                and hasattr(past_key_value, "update")
                and self.num_kv_heads is not None and self.head_dim is not None):
            bsz, seq_len = hidden_states.shape[0], hidden_states.shape[1]
            dummy = torch.zeros(bsz, self.num_kv_heads, seq_len, self.head_dim,
                                dtype=hidden_states.dtype,
                                device=hidden_states.device)
            past_key_value.update(dummy, dummy, self.layer_idx, {})
        outputs = (hidden_states,)
        if output_attentions:
            outputs += (None,)
        if use_cache:
            outputs += (past_key_value,)
        return outputs

## HOOK C — where this goes

Not in the control loop: **inside the model**, once per episode.

```
policy.reset()
├─ first policy.step(...) of the episode
│    └─ forward hooks record 1 − cos(in, out) per layer   ← MEASURE
├─ rank, pick N, swap those modules for BypassDecoderLayer ← APPLY
└─ every subsequent step runs the pruned stack
```

Finding the layer stack is the only backbone-specific part, and it is
handled by walking candidate attribute paths rather than hard-coding one —
Emu3 exposes it at `model.model.layers`, while OpenVLA wraps a Llama inside
Prismatic so it sits at `model.language_model.model.layers`. A hard-coded
path that silently misses is how an "unpruned" run gets reported as pruned.

In [ ]:
import math
import numpy as np


def find_decoder_layers(model):
    """Locate the decoder stack across the wrappers different VLAs use."""
    candidates = (
        ("model", "layers"),
        ("language_model", "model", "layers"),
        ("language_model", "layers"),
        ("model", "language_model", "layers"),
        ("layers",),
    )
    for path in candidates:
        node = model
        for attr in path:
            node = getattr(node, attr, None)
            if node is None:
                break
        if isinstance(node, torch.nn.ModuleList) and len(node) > 0:
            return node
    return None


def measure_redundancy_with_hooks(layers, run_fn):
    """Per-layer 1 - cos(in, out), captured while run_fn() executes.

    Only the FIRST call of each layer is recorded. Generation calls every
    layer once per decoded token; the prefill -- the first call -- is the
    one that sees the whole prompt. Averaging in single-token decode steps
    would measure something else.

    Must run with the stack UNPRUNED: a bypassed layer has input == output,
    so it would report ~0 redundancy and rank itself most-redundant forever.
    """
    scores = [None] * len(layers)
    handles = []

    def make_hook(idx):
        def hook(module, args, kwargs, output):
            if scores[idx] is not None:
                return
            inp = args[0] if args else kwargs.get("hidden_states")
            out = output[0] if isinstance(output, tuple) else output
            if inp is None or out is None or not torch.is_tensor(inp):
                return
            cos = torch.nn.functional.cosine_similarity(
                inp.float(), out.float(), dim=-1)
            scores[idx] = float(1.0 - cos.mean().item())
        return hook

    for i, layer in enumerate(layers):
        handles.append(layer.register_forward_hook(make_hook(i), with_kwargs=True))
    try:
        run_fn()
    finally:
        for h in handles:
            h.remove()
    return None if any(s is None for s in scores) else [float(s) for s in scores]


def rank_layers(importance, min_layer=0.5, min_gap=1):
    """Eligible layers, most-redundant first, gap-respecting."""
    scores = np.asarray(importance, dtype=np.float32)
    n = int(scores.shape[0])
    start = int(math.floor(np.clip(min_layer, 0.0, 1.0) * n))
    candidates = list(range(start, n)) or list(range(n))
    ranked = sorted(candidates, key=lambda i: (float(scores[i]), i))
    ordered = []
    for i in ranked:                       # gap-respecting greedy first
        if any(abs(i - prev) <= min_gap for prev in ordered):
            continue
        ordered.append(i)
    for i in ranked:                       # then fill in the rest
        if i not in ordered:
            ordered.append(i)
    return ordered

In [ ]:
class StaticDepthPruner:
    """Calibrate once per episode, then bypass the N most redundant layers."""

    def __init__(self, model, prune=8, min_layer=0.5, min_gap=1):
        self.model, self.prune = model, int(prune)
        self.min_layer, self.min_gap = min_layer, min_gap
        self._originals, self._active, self._done = {}, (), False

    def layers(self):
        return find_decoder_layers(self.model)

    def restore(self):
        layers = self.layers()
        if layers is None or not self._originals:
            return
        for idx, layer in self._originals.items():
            layers[idx] = layer
        self._originals, self._active = {}, ()

    def _head_shape(self):
        """Prismatic-style wrappers nest the LLM config; the outer config
        has no attention shape and a wrong head_dim makes a KV placeholder
        the cache cannot concatenate."""
        cfg = getattr(self.model, "config", None)
        for attr in ("text_config", "llm_config", "language_model_config"):
            sub = getattr(cfg, attr, None)
            if sub is not None and getattr(sub, "num_attention_heads", None):
                cfg = sub
                break
        n_heads = getattr(cfg, "num_attention_heads", None)
        hidden = getattr(cfg, "hidden_size", None)
        kv = getattr(cfg, "num_key_value_heads", None) or n_heads
        head_dim = getattr(cfg, "head_dim", None) or (
            (hidden // n_heads) if (hidden and n_heads) else None)
        return kv, head_dim

    def apply(self, indices):
        layers = self.layers()
        if layers is None:
            return
        valid = tuple(i for i in sorted({int(x) for x in indices})
                      if 0 <= i < len(layers))
        self.restore()          # restore first, or bypasses accumulate
        if not valid:
            return
        kv, head_dim = self._head_shape()
        for idx in valid:
            self._originals[idx] = layers[idx]
            layers[idx] = BypassDecoderLayer(idx, num_kv_heads=kv,
                                             head_dim=head_dim)
        self._active = valid

    def calibrate_on(self, run_fn):
        """Call with the model's real first forward of the episode."""
        if self._done:
            return run_fn()
        layers = self.layers()
        if layers is None:
            raise RuntimeError(
                "decoder stack not found -- refusing to report an unpruned "
                "run as pruned. Add this model's attribute path to "
                "find_decoder_layers.")
        captured = {}
        importance = measure_redundancy_with_hooks(
            layers, lambda: captured.setdefault("out", run_fn()))
        if importance is not None:
            self.apply(rank_layers(importance, self.min_layer, self.min_gap)[:self.prune])
            print(f"[depth] bypassing {list(self._active)} of {len(layers)}")
        self._done = True
        return captured.get("out")

    def reset_episode(self):
        self._done = False
        self.restore()

## What this assumes about the architecture — check before porting

This is the most architecture-dependent of the three methods. Foveation
touches an image and action repeat touches an array; this one reaches inside
the model, so it inherits assumptions the other two do not.

| assumption | fails when | symptom / what to do |
|---|---|---|
| **the stack is a flat `ModuleList` of interchangeable decoder layers** | the model interleaves a different layer type — e.g. gated cross-attention blocks in Flamingo-style architectures | bypassing a cross-attention block removes the *vision pathway*, not redundant compute. Inspect the module list and exclude any layer type that is not a plain self-attention block |
| **the decoder dominates the step** | the LLM runs **once** per step and a small head (MLP, LSTM, diffusion/flow) produces the action | there is no autoregressive decode to shrink, so the saving is only on prefill and is much smaller. Profile first; the intervention may not be worth running |
| **cache is a per-layer list indexed by `layer_idx`** (`DynamicCache`) | the model uses a static or hybrid cache with pre-allocated shape, e.g. Gemma2's sliding-window `HybridCache` | the zero placeholder may not be the shape the cache expects. Verify cached vs uncached generation produce identical tokens *on that model* |
| **KV shape is `(batch, kv_heads, seq, head_dim)`** | attention variants that do not store K and V in that layout (e.g. latent-compressed attention) | the placeholder cannot be concatenated. Read the model's own attention code before trusting `_head_shape()` |
| **bypassing preserves sequence semantics** | layers carry per-layer positional or rotary state that later layers depend on | rare, but shows up as coherent-looking output that ignores the instruction |

None of these produce a clean error. Each produces a lower success rate,
which is indistinguishable from "depth pruning does not work on this
backbone" — the exact claim the experiment is trying to test. That is why
the verification cell below is not optional.

## Wiring it into a policy

```python
class Policy:
    def __init__(self, model, prune=8):
        self.model = model
        self.depth = StaticDepthPruner(model, prune=prune)

    def reset(self):
        self.depth.reset_episode()      # re-measure on the unpruned stack

    def step(self, image, instruction):
        inputs = self.preprocess(image, instruction)
        run = lambda: self.model.generate(**inputs)
        out = self.depth.calibrate_on(run)     # measures on call 1, then pruned
        return self.detokenise(out)
```

One episode therefore runs its first step unpruned — 1 of up to ~230 — and
every step after that on the pruned stack.

## Check: does a pruned model with cache still decode correctly?

This is the check that matters. If the KV placeholder is wrong, `generate()`
with a cache diverges from uncached greedy decoding, and the only symptom is
a lower success rate weeks later.

The cell below builds a small stack in plain PyTorch and verifies the bypass
behaves as a true identity on hidden states while keeping the cache
contiguous. On a real checkpoint, run the same comparison with
`model.generate(..., use_cache=True)` against `use_cache=False`; they must
produce **identical token ids**, not merely similar ones.

In [ ]:
class _FakeLayer(torch.nn.Module):
    def __init__(self, idx, hidden=16, kv_heads=2, head_dim=8):
        super().__init__()
        self.idx, self.kv_heads, self.head_dim = idx, kv_heads, head_dim
        self.lin = torch.nn.Linear(hidden, hidden)
        # Layer 3 and 5 are near-identity: they should rank most redundant.
        with torch.no_grad():
            if idx in (3, 5):
                self.lin.weight.copy_(torch.eye(hidden) + 1e-4)
                self.lin.bias.zero_()

    def forward(self, hidden_states, past_key_value=None, use_cache=False, **kw):
        out = hidden_states + 0.02 * self.lin(hidden_states)
        if use_cache and past_key_value is not None:
            b, s = hidden_states.shape[0], hidden_states.shape[1]
            d = torch.zeros(b, self.kv_heads, s, self.head_dim)
            past_key_value.update(d, d, self.idx, {})
        return (out, past_key_value) if use_cache else (out,)


class _FakeCache:
    def __init__(self):
        self.slots = []
    def update(self, k, v, idx, _):
        while len(self.slots) <= idx:
            self.slots.append(None)
        if self.slots[idx] is not None:
            raise IndexError("layer wrote twice")
        self.slots[idx] = (k, v)
        return k, v


class _FakeModel(torch.nn.Module):
    def __init__(self, n=8, hidden=16):
        super().__init__()
        self.layers = torch.nn.ModuleList([_FakeLayer(i, hidden) for i in range(n)])
        class _Cfg: num_attention_heads = 2; hidden_size = hidden; num_key_value_heads = 2
        self.config = _Cfg()
    def forward(self, x, use_cache=False):
        cache = _FakeCache() if use_cache else None
        for layer in self.layers:
            x = layer(x, past_key_value=cache, use_cache=use_cache)[0]
        return x, cache


torch.manual_seed(0)
model = _FakeModel()
x = torch.randn(1, 5, 16)

print("[1] the stack is found:", find_decoder_layers(model) is not None)

imp = measure_redundancy_with_hooks(model.layers, lambda: model(x))
print("[2] per-layer 1-cos(in,out):", [round(v, 5) for v in imp])

order = rank_layers(imp, min_layer=0.5, min_gap=1)
print("[3] ranked (most redundant first, back half only):", order)
assert min(order[:2]) >= 4, "early layers must never be eligible"

pruner = StaticDepthPruner(model, prune=2)
pruner.calibrate_on(lambda: model(x))
bypassed = pruner._active
print("[4] bypassed:", list(bypassed))

# A bypassed layer must be an exact identity on hidden states...
layer = model.layers[bypassed[0]]
probe = torch.randn(1, 3, 16)
assert torch.equal(layer(probe)[0], probe)
print("[5] bypassed layer is an exact identity on hidden states")

# ...and the cache must stay contiguous with no gaps.
_, cache = model(x, use_cache=True)
print("[6] cache slots filled:", sum(s is not None for s in cache.slots),
      "of", len(model.layers))
assert all(s is not None for s in cache.slots), "gap in the KV cache"

pruner.restore()
assert not isinstance(model.layers[bypassed[0]], BypassDecoderLayer)
print("[7] restore() puts the real modules back")
print("\nALL CHECKS PASSED")

## Caveats to carry into the results table

* **Calibrate on the unpruned stack, every episode.** A bypassed layer has
  input == output, so if it is measured while already bypassed it scores ~0
  redundancy and locks itself in permanently.
* **How much depth a backbone can spare is a property of the backbone**, and
  it varies enormously. Measured at the identical rule and ratio (8 of 32
  layers): Emu3 lost 10 points, Llama-2 lost 46. Gemma2 (26 layers) lost
  accuracy on 3 of 4 tasks with a **single** layer bypassed. Do not carry a
  value of N across backbones — measure the curve.
* **Verify the speedup actually happened.** Bypassing layers should reduce
  ms/call by roughly N/total. If success drops and latency did not move,
  the layers were not really bypassed — check that `find_decoder_layers`
  found the right stack.